# Customer Churn Segmentation & Rule-Based Risk Analysis

**Task:** Data Analytics – Task 3: Segmentation and Recommendation Model  
**Objective:** Segment customers by behavior and create a transparent, reproducible churn-risk score.

### Decision question
Which customer groups should a fictional subscription team contact first to reduce avoidable churn?

> **Important:** This is a small, fictional sample of 15 customers. The score is a prioritization tool, not a causal model or a production prediction model.


## 1. Approach

The score uses four account/behavior features:

- **ContractType:** Monthly contract = +2
- **SupportTickets:** 4 or more = +2; 2–3 = +1
- **TenureMonths:** 12 or less = +2; 13–24 = +1
- **MonthlyCharges:** ₹100 or more = +1

`Churn` is **not** used to calculate the score. It is used afterward only to evaluate how the rule behaves on the supplied sample.

### Risk bands
- **0–1:** Low Risk
- **2–3:** Medium Risk
- **4+:** High Risk


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Load the supplied dataset.
df = pd.read_csv("customer_churn_sample (1).csv")

print("Shape:", df.shape)
display(df.head())


## 2. Data quality checks

In [ ]:
# Missing values
print("Missing values by column:")
display(df.isna().sum().to_frame("missing_values"))

# Duplicate rows
print("Duplicate rows:", df.duplicated().sum())

# Duplicate customer IDs
print("Duplicate CustomerID values:", df["CustomerID"].duplicated().sum())

# Basic range checks
print("Tenure minimum:", df["TenureMonths"].min())
print("Tenure maximum:", df["TenureMonths"].max())
print("Monthly charge minimum:", df["MonthlyCharges"].min())
print("Monthly charge maximum:", df["MonthlyCharges"].max())
print("Support tickets minimum:", df["SupportTickets"].min())
print("Support tickets maximum:", df["SupportTickets"].max())


### Data-quality conclusion

The supplied sample has 15 rows and 11 columns. The checks above are used to identify missing values, duplicate rows/customer IDs, and unusual numeric ranges before analysis. No rows are silently deleted.


## 3. Create the reproducible churn-risk score

In [ ]:
def calculate_risk_score(row):
    score = 0

    # Contract behavior
    if str(row["ContractType"]).strip().lower() == "monthly":
        score += 2

    # Recent support activity
    if row["SupportTickets"] >= 4:
        score += 2
    elif row["SupportTickets"] >= 2:
        score += 1

    # Customer tenure
    if row["TenureMonths"] <= 12:
        score += 2
    elif row["TenureMonths"] <= 24:
        score += 1

    # Monthly charge
    if row["MonthlyCharges"] >= 100:
        score += 1

    return score

def risk_level(score):
    if score <= 1:
        return "Low"
    elif score <= 3:
        return "Medium"
    return "High"

analysis = df.copy()
analysis["RiskScore"] = analysis.apply(calculate_risk_score, axis=1)
analysis["RiskLevel"] = analysis["RiskScore"].apply(risk_level)

display(
    analysis[
        ["CustomerID", "TenureMonths", "MonthlyCharges",
         "SupportTickets", "ContractType", "Churn",
         "RiskScore", "RiskLevel"]
    ]
)


## 4. Customer segments

In [ ]:
def segment_name(level):
    if level == "High":
        return "High-Risk Priority"
    elif level == "Medium":
        return "Medium-Risk Monitor"
    return "Low-Risk Maintain"

analysis["Segment"] = analysis["RiskLevel"].apply(segment_name)

segment_summary = (
    analysis.groupby("RiskLevel", observed=False)
    .agg(
        Customers=("CustomerID", "count"),
        Churned=("Churn", lambda x: (x == "Yes").sum())
    )
    .reset_index()
)

segment_summary["ChurnRate"] = (
    segment_summary["Churned"] / segment_summary["Customers"] * 100
).round(1)

segment_summary = segment_summary.set_index("RiskLevel").reindex(
    ["High", "Medium", "Low"]
).reset_index()

display(segment_summary)


### Segment interpretation

**High-Risk Priority:** Customers with the strongest combination of short tenure, support activity, monthly contracts and/or higher monthly charges. Contact these customers first.

**Medium-Risk Monitor:** Customers showing some risk signals but not enough to be in the highest-priority group. Monitor support issues and consider targeted retention outreach.

**Low-Risk Maintain:** Customers with fewer risk signals. Maintain service quality and avoid unnecessary intensive retention campaigns.


## 5. Visualize the segment results

In [ ]:
plot_data = segment_summary.dropna(subset=["RiskLevel"]).copy()

plt.figure(figsize=(7, 4))
plt.bar(plot_data["RiskLevel"], plot_data["ChurnRate"])
plt.xlabel("Risk level")
plt.ylabel("Observed churn rate (%)")
plt.title("Observed Churn Rate by Rule-Based Risk Level")
plt.ylim(0, 110)
plt.show()


## 6. Edge-case validation

The rule boundaries are:

- Score **1** → Low
- Score **2** → Medium
- Score **3** → Medium
- Score **4** → High

The following test checks those exact boundaries.


In [ ]:
test_scores = pd.DataFrame({
    "RiskScore": [1, 2, 3, 4, 5],
})
test_scores["RiskLevel"] = test_scores["RiskScore"].apply(risk_level)

expected = ["Low", "Medium", "Medium", "High", "High"]
assert test_scores["RiskLevel"].tolist() == expected

print("Boundary tests passed.")
display(test_scores)


## 7. Key findings

In [ ]:
# Customer counts and churn rates
summary = (
    analysis.groupby("RiskLevel", observed=False)
    .agg(
        Customers=("CustomerID", "count"),
        Churned=("Churn", lambda x: (x == "Yes").sum())
    )
)

summary["ChurnRate"] = (summary["Churned"] / summary["Customers"] * 100).round(1)
summary = summary.reindex(["High", "Medium", "Low"])

display(summary)


### Findings from the supplied sample

- **High Risk:** 5 customers; 5 churned → **100% observed churn rate**
- **Medium Risk:** 4 customers; 2 churned → **50% observed churn rate**
- **Low Risk:** 6 customers; 0 churned → **0% observed churn rate**

The pattern is directionally useful for prioritization in this sample. However, the sample contains only 15 customers, so these percentages are unstable and should not be generalized to a larger customer population without more data.

**No causal claim is made:** the analysis shows an association between the rule-based segments and observed churn in this sample; it does not prove that any individual factor causes churn.


## 8. Recommendation memo

**Audience:** Fictional subscription retention team

**Decision:** Contact High-Risk customers first, followed by Medium-Risk customers. Low-Risk customers should mainly receive normal service and loyalty/engagement communication.

**Supporting metric:** All 5 High-Risk customers in the supplied sample churned, compared with 0 of 6 Low-Risk customers.

**Uncertainty:** The dataset contains only 15 customers. The observed 100%, 50%, and 0% rates are therefore highly sensitive to individual customers and should not be treated as population estimates.

**Next measurement:** In a larger follow-up sample, track churn rate, support-ticket resolution, contract upgrades, and retention among customers contacted by risk segment. Compare contacted versus non-contacted groups before concluding that the intervention improved retention.


## 9. Final takeaway

The rule-based model provides a transparent way to prioritize retention effort:

1. **High Risk → contact first**
2. **Medium Risk → monitor and contact selectively**
3. **Low Risk → maintain normal engagement**

The model is intentionally explainable and reproducible rather than a black-box machine-learning model.
